# Condition C: Combined Checker-Repair Sweep

**What this artifact does.** Condition C tests whether *checker-broadened localization* (flag every content-invariant span with a deterministic checker, not just the one corrupted span) combined with *iterative repair-and-reverify* (re-run the checker after each repair pass, stop when it certifies zero flags, up to `MAX_PASSES=3`) beats two earlier conditions tested in isolation:

- **C1** (iter_2): checker-broadened localization, but only a single uncorrected repair pass.
- **C2** (iter_3): the same narrow, single-span localization as baselines B/D, but iterated with re-verification.

Condition C combines both mechanisms: on every pass it re-runs the checker's `detect_all()` on the **current** candidate translation (not a pre-computed span), repairs whatever it flags, and stops on a *certificate* (zero flags). Because the checker's absence-vs-source heuristic can flag a negation that was **deleted** entirely (something a narrow oracle-span localizer structurally cannot even point at), Condition C is also the first condition able to *attempt* `negation_polarity_flip` deletion rows — logged separately since there is no corrupted span to check the absence of.

**What this notebook does.** The original `method.py` pipeline calls `google/gemma-3-12b-it` via OpenRouter (731 real LLM calls, ~70 min, real API key required) and the `Unbabel/wmt22-cometkiwi-da` COMET checkpoint. Neither is available in a notebook/Colab context, so this demo reuses the **already-generated repair outputs** (`predict_c`) from a curated subset of the real 560-row sweep, and re-runs the exact same deterministic verification code from `method.py`/`checker.py` on them: the four-category checker (`checker.detect_all`), the certificate check, the edit-distance/edit-volume functions, and the pooled-metrics aggregation. Nothing here is mocked — every number recomputed below is produced by the artifact's own unmodified functions running on real model outputs; only the (expensive, key-gated) generation step itself is replaced by its already-recorded result.

Data is loaded from a small curated JSON (`mini_demo_data.json`, 60 rows: 12 natural + 48 injected-error rows spanning all 8 language×category cells) hosted on GitHub, with a local fallback.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# The demo only needs stdlib (re, dataclasses, collections, json) plus matplotlib
# for the results plot -- matplotlib is pre-installed on Colab, so it is the only
# package behind the google.colab guard. No non-Colab packages are required: the
# original method.py's other dependencies (aiohttp, loguru, stanza, comet, psutil)
# are only needed for the LLM-calling / COMET-scoring / resource-limiting parts of
# the pipeline, which this demo does not re-run (see markdown above).
if 'google.colab' not in sys.modules:
    _pip('matplotlib==3.10.0')


In [ ]:
# Imports -- copied from method.py's original import block, minus the pieces
# only needed for the live LLM-calling / resource-limiting parts we skip here
# (asyncio, resource, psutil, gc, checker.py's own llm_client import).
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from dataclasses import dataclass

import matplotlib.pyplot as plt


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-e9bf19-isolating-what-fixes-failed-span-editing/main/round-4/experiment-1/demo/mini_demo_data.json"

import os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
data = load_data()

natural_rows = data["datasets"][0]["examples"]
injected_rows = data["datasets"][1]["examples"]
run_metadata = data["metadata"]

print(f"Loaded {len(natural_rows)} natural rows and {len(injected_rows)} injected-error rows")
print(f"Repair model (original run): {run_metadata['repair_model']}")
print(f"Language pairs: {run_metadata['language_pairs']}")


## Config

All tunable parameters from the original `method.py`, set to the values the real run used (`MAX_PASSES=3`, `SEED=42`, the exact category-name mappings). `EXCLUDED_CELLS` is the one config value that in the original run is *computed* by `validate_checker()` against a held-out validation fold before the sweep starts (Phase 1 of `main()`); the held-out fold is not part of this demo's curated data, so here we reuse the real run's already-computed exclusion result (persisted in `run_metadata["checker_validation"]["excluded_cells"]`) rather than re-deriving it -- this is a data reuse, not a code change, and it is exactly how the checker gets constructed for the repair sweep in `main()` (`checker = Checker(excluded_cells=excluded)`).

In [ ]:
# --- config, copied verbatim from method.py's Config section -----------------
SEED = 42
LANG_PAIRS = ["en-ru_RU", "en-uk_UA"]
LANG_KEY_OF_PAIR = {"en-ru_RU": "ru_RU", "en-uk_UA": "uk_UA"}
LANG_NAME_OF_PAIR = {"en-ru_RU": "Russian", "en-uk_UA": "Ukrainian"}

MODEL = "google/gemma-3-12b-it"
MAX_PASSES = 3
BOOTSTRAP_ITERS = 1000  # kept at the original value -- bootstrap CIs are cheap even on 60 rows

CATEGORY_MAP_INJECTED_TO_CHECKER = {
    "named_entity_swap": "named_entity",
    "number_unit_date_alteration": "number_unit_date",
    "negation_polarity_flip": "negation_polarity",
    "quantifier_substitution": "quantifier_scope",
}
CHECKER_TO_INJECTED_CATEGORY = {v: k for k, v in CATEGORY_MAP_INJECTED_TO_CHECKER.items()}

HYGIENE_REGEX = re.compile(
    r"__BLANK__|__[A-Z]+__|\bCorrected words\b|\bHere is the corrected\b|\bHere's the corrected\b",
    re.IGNORECASE,
)

# reused from the real run's Phase-1 checker validation (see markdown above)
EXCLUDED_CELLS = {tuple(c.split("|")) for c in run_metadata["checker_validation"]["excluded_cells"]}
print("EXCLUDED_CELLS (lang, category):", EXCLUDED_CELLS)


## The checker (`checker.py`, copied verbatim)

This is the deterministic, four-category content-invariant checker the whole condition is built around: `named_entity` (Stanza NER -- not invoked in this demo since both language cells are excluded, see `EXCLUDED_CELLS` above, so the lazy Stanza import inside `NamedEntityDetector._get_pipeline` never fires), `number_unit_date` (script-independent digit regex), `negation_polarity` (per-language cue-word list, including the absence-vs-source deletion heuristic), and `quantifier_scope` (closed-class quantifier word list). `Checker.detect_all()` is the function Condition C re-runs on every repair pass -- an empty return is the "certificate".

In [ ]:
# checker.py -- copied verbatim (see workspace_path/checker.py for the original file)

CATEGORIES = (
    "named_entity",
    "number_unit_date",
    "negation_polarity",
    "quantifier_scope",
)

SUPPORTED_LANGS = ("ru_RU", "uk_UA")

# --- number/unit/date -------------------------------------------------------
# Script-independent digit-sequence regex + an optional attached unit/word
# token (letters immediately following, any script). Boundary guard: a digit
# run immediately preceded by '@' (handle-like token, e.g. @user12) is
# excluded.
_NUMBER_RE = re.compile(
    r"(?<![@\w])\d+(?:[.,]\d+)*\s?[A-Za-zА-Яа-яЁёІіЇїЄєҐґ]{0,15}\b"
)

# --- negation cue lists -------------------------------------------------
NEGATION_CUES = {
    "ru_RU": [r"не", r"нет", r"ни"],
    "uk_UA": [r"не", r"ні", r"жодн\w*"],
}

# English negation cues, used only to decide whether an ALIGNED source
# sentence plausibly carries a negation the hypothesis sentence is missing
# (see detect_negation_polarity: the injected-data negation corruption is a
# DELETION of the target-language marker, so localizing it requires noticing
# an ABSENCE relative to source, not just matching a present cue word).
_EN_NEGATION_RE = re.compile(
    r"\bnot\b|n't\b|\bnever\b|\bno\b|\bnothing\b|\bnobody\b|\bnone\b|\bneither\b|\bnor\b|\bwithout\b",
    re.IGNORECASE,
)

# --- quantifier closed-class word lists ---------------------------------
QUANTIFIER_WORDS = {
    "ru_RU": [
        "все", "всех", "всем", "всеми", "весь", "вся", "всё",
        "каждый", "каждая", "каждое", "каждые", "каждого", "каждой",
        "некоторые", "некоторых", "многие", "многих", "мало",
        "несколько", "нескольких", "любой", "любая", "любое",
        "никто", "ничто", "никакой",
    ],
    "uk_UA": [
        "всі", "весь", "вся", "все", "усі", "увесь",
        "кожен", "кожна", "кожне", "кожні", "кожного", "кожної",
        "деякі", "деяких", "багато", "багатьох", "мало",
        "декілька", "кількох", "будь-який", "будь-яка", "будь-яке",
        "ніхто", "ніщо", "жоден",
    ],
}


@dataclass
class Span:
    start: int
    end: int
    text: str
    category: str


def _regex_spans(pattern: re.Pattern, text: str, category: str) -> list[Span]:
    return [Span(m.start(), m.end(), m.group(0), category) for m in pattern.finditer(text)]


def detect_number_unit_date(hyp_text: str) -> list[Span]:
    return _regex_spans(_NUMBER_RE, hyp_text, "number_unit_date")


_SENTENCE_SPLIT_RE = re.compile(r"[^.!?\n]*[.!?\n]|[^.!?\n]+$")


def _sentence_spans(text: str) -> list[tuple[int, int]]:
    """Char (start, end) offsets for each sentence-like chunk of text (split
    on .!?/newline, terminator kept with the preceding chunk)."""
    spans = []
    for m in _SENTENCE_SPLIT_RE.finditer(text):
        if m.group(0).strip():
            spans.append((m.start(), m.end()))
    return spans or [(0, len(text))]


def detect_negation_polarity(source_text: str, hyp_text: str, lang: str) -> list[Span]:
    """Per-language cue-word match, word-boundary, applied per SENTENCE (not
    the whole multi-sentence hypothesis document): a sentence fires when it
    contains EXACTLY ONE negation marker (Slavic/Czech negative-concord
    languages allow multiple co-occurring negation markers within a sentence,
    so removing just one does not reliably flip polarity unless it is the
    only one in that sentence -- the dossier's documented caveat, scoped here
    to the sentence it actually applies to rather than the whole document).

    A second signal handles the DELETION case (the injected negation-flip
    corruption removes the target-language marker entirely, leaving nothing
    to text-match at that position): a hyp sentence with ZERO negation cues
    whose position-aligned source sentence (coarse index-proportional
    alignment -- documents are similar length, exact sentence alignment is
    not available) DOES contain an English negation cue is flagged as a
    suspected negation-deletion site (the whole hyp sentence is the span,
    since there is no token to point to)."""
    cues = NEGATION_CUES[lang]
    combined = re.compile(r"\b(?:" + "|".join(cues) + r")\b", re.IGNORECASE)
    hyp_sents = _sentence_spans(hyp_text)
    src_sents = _sentence_spans(source_text) if source_text else []
    out: list[Span] = []
    for idx, (sent_start, sent_end) in enumerate(hyp_sents):
        sentence = hyp_text[sent_start:sent_end]
        matches = list(combined.finditer(sentence))
        if len(matches) == 1:
            m = matches[0]
            out.append(Span(sent_start + m.start(), sent_start + m.end(), m.group(0), "negation_polarity"))
        elif len(matches) == 0 and src_sents:
            src_idx = min(int(idx * len(src_sents) / max(1, len(hyp_sents))), len(src_sents) - 1)
            src_st, src_en = src_sents[src_idx]
            if _EN_NEGATION_RE.search(source_text[src_st:src_en]):
                out.append(Span(sent_start, sent_end, sentence, "negation_polarity"))
    return out


def detect_quantifier_scope(hyp_text: str, lang: str) -> list[Span]:
    words = sorted(QUANTIFIER_WORDS[lang], key=len, reverse=True)
    combined = re.compile(r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b", re.IGNORECASE)
    return _regex_spans(combined, hyp_text, "quantifier_scope")


class NamedEntityDetector:
    """Wraps a per-language Stanza NER pipeline. Lazily initialized (Stanza
    pipeline construction loads model weights, so this is done once per
    language and reused for every row)."""

    def __init__(self):
        self._pipelines: dict[str, object] = {}

    def _get_pipeline(self, lang: str):
        if lang not in self._pipelines:
            import stanza

            stanza_lang = {"ru_RU": "ru", "uk_UA": "uk"}[lang]
            self._pipelines[lang] = stanza.Pipeline(
                lang=stanza_lang,
                processors="tokenize,ner",
                use_gpu=False,
                verbose=False,
                download_method=None,
            )
        return self._pipelines[lang]

    def detect(self, hyp_text: str, lang: str) -> list[Span]:
        nlp = self._get_pipeline(lang)
        doc = nlp(hyp_text)
        spans = []
        for ent in doc.ents:
            spans.append(Span(ent.start_char, ent.end_char, ent.text, "named_entity"))
        return spans


class Checker:
    """Runs the four category detectors, respecting an exclusion list of
    (lang, category) cells that failed Phase-1 validation."""

    def __init__(self, excluded_cells: set[tuple[str, str]] | None = None):
        self.excluded_cells = excluded_cells or set()
        self._ner = NamedEntityDetector()

    def detect_category(self, category: str, source_text: str, hyp_text: str, lang: str) -> list[Span]:
        if category == "named_entity":
            return self._ner.detect(hyp_text, lang)
        if category == "number_unit_date":
            return detect_number_unit_date(hyp_text)
        if category == "negation_polarity":
            return detect_negation_polarity(source_text, hyp_text, lang)
        if category == "quantifier_scope":
            return detect_quantifier_scope(hyp_text, lang)
        raise ValueError(f"Unknown category: {category}")

    def detect_all(self, source_text: str, hyp_text: str, lang: str, *, respect_exclusions: bool) -> list[Span]:
        spans: list[Span] = []
        for cat in CATEGORIES:
            if respect_exclusions and (lang, cat) in self.excluded_cells:
                continue
            spans.extend(self.detect_category(cat, source_text, hyp_text, lang))
        return spans

    def is_flagged(self, category: str, source_text: str, hyp_text: str, lang: str, start: int, end: int) -> bool:
        """True if any span from this category's detector overlaps [start, end)."""
        for s in self.detect_category(category, source_text, hyp_text, lang):
            if s.start < end and start < s.end:
                return True
        return False


checker = Checker(excluded_cells=EXCLUDED_CELLS)
print("Checker constructed with excluded_cells:", checker.excluded_cells)


## Edit-distance / edit-volume helpers (`method.py`, copied verbatim)

Condition C measures how much a repair pass changed the text with `char_edit_distance` (used per-pass in the pass log) and `edit_volume` (word-level edit distance normalized by original length, used in the pooled natural-row metrics). `invariant_multiset` is the function `compute_fix_and_regression` uses to count invariant-bearing spans the checker finds -- reused below directly since it only needs `(source, text, lang, checker)`, none of which requires the corrupted-span ground truth.

In [ ]:
# method.py -- edit-distance / edit-volume / invariant-multiset helpers, copied verbatim

def char_edit_distance(a: str, b: str) -> int:
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[m]


def word_edit_distance(a: str, b: str) -> int:
    aw, bw = a.split(), b.split()
    n, m = len(aw), len(bw)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if aw[i - 1] == bw[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[m]


def edit_volume(original: str, edited: str) -> float:
    n = max(1, len(original.split()))
    return word_edit_distance(original, edited) / n


def invariant_multiset(source_text: str, text: str, lang: str, checker: Checker) -> Counter:
    spans = checker.detect_all(source_text, text, lang, respect_exclusions=True)
    return Counter((s.category, s.text.strip().lower()) for s in spans)


def has_leakage(text: str) -> bool:
    return bool(HYGIENE_REGEX.search(text))


## Reproducing the certificate check on real repaired outputs

This is the exact final step of `run_condition_c_core()` in `method.py`:

```python
flags_final = checker.detect_all(source, current, lang, respect_exclusions=True)
certificate_achieved = len(flags_final) == 0
```

`current` there is the text after up to `MAX_PASSES` repair passes, which is exactly the `predict_c` field persisted for every row. Running `checker.detect_all()` (the same unmodified function above) on each row's real `predict_c` output reproduces the `certificate_achieved` / `n_flags_remaining_final` values the real sweep computed and logged. The `input` field in the loaded data is prefixed with `[language_pair]` or `[language_pair/category]` (as written by `method.py`'s own `natural_example`/`injected_example` output-assembly functions) -- stripped below to recover the raw source sentence `detect_all` expects.

In [ ]:
def strip_tag_prefix(tagged_input: str) -> str:
    """Undo method.py's f"[{tag}] {source}" / f"[{tag}/{cat}] {source}" prefixing."""
    if tagged_input.startswith("[") and "] " in tagged_input:
        return tagged_input.split("] ", 1)[1]
    return tagged_input


n_cert_match = 0
n_rows_checked = 0
recomputed_natural = []
for row in natural_rows:
    lang = LANG_KEY_OF_PAIR[row["metadata_language_pair"]]
    source = strip_tag_prefix(row["input"])
    predict_c = row["predict_c"]
    flags_final = checker.detect_all(source, predict_c, lang, respect_exclusions=True)
    cert_recomputed = len(flags_final) == 0
    ev_recomputed = round(edit_volume(row["output"], predict_c), 4)
    recomputed_natural.append({
        "row_id": row["metadata_row_id"], "language_pair": row["metadata_language_pair"],
        "n_flags_remaining_final": len(flags_final), "certificate_recomputed": cert_recomputed,
        "certificate_logged": row["metadata_c_certificate_achieved"],
        "edit_volume_recomputed": ev_recomputed, "edit_volume_logged": row["metadata_c_edit_volume"],
    })
    n_rows_checked += 1
    n_cert_match += int(cert_recomputed == row["metadata_c_certificate_achieved"])

recomputed_injected = []
for row in injected_rows:
    lang = LANG_KEY_OF_PAIR[row["metadata_language_pair"]]
    source = strip_tag_prefix(row["input"])
    predict_c = row["predict_c"]
    flags_final = checker.detect_all(source, predict_c, lang, respect_exclusions=True)
    cert_recomputed = len(flags_final) == 0
    recomputed_injected.append({
        "row_id": row["metadata_row_id"], "language_pair": row["metadata_language_pair"],
        "category": row["metadata_invariant_category"],
        "is_negation_deletion_row": row["metadata_is_negation_deletion_row"],
        "n_flags_remaining_final": len(flags_final), "certificate_recomputed": cert_recomputed,
        "certificate_logged": row["metadata_c_certificate_achieved"],
        "fixed": row["metadata_fixed"], "true_regression": row["metadata_true_regression"],
    })
    n_rows_checked += 1
    n_cert_match += int(cert_recomputed == row["metadata_c_certificate_achieved"])

print(f"Certificate check reproduced on {n_rows_checked} rows ({len(natural_rows)} natural + {len(injected_rows)} injected)")
print(f"Recomputed certificate matches the real run's logged value on {n_cert_match}/{n_rows_checked} rows")


## Pooled metrics by category (`summarize_condition_c`, same aggregation logic)

`method.py`'s `summarize_condition_c()` builds a `category_table`: one row per `(injected category, language pair)` cell with `fix_rate`, `true_regression_rate`, and `certificate_rate`. On this 48-row injected subset (6 rows per cell x 8 cells) the same grouping and averaging is applied directly to the loaded `metadata_fixed` / `metadata_true_regression` / `metadata_c_certificate_achieved` fields (the ground-truth `fixed`/`true_regression` verdicts themselves require `metadata_corrupted_span` and `metadata_clean_target_text`, which are not part of the persisted example schema -- `compute_fix_and_regression()` computed them once inside the real sweep and `method.py` writes only the resulting boolean verdict into `metadata_fixed`, so this cell aggregates that verdict rather than re-deriving it from scratch).

In [ ]:
# same grouping logic as method.py's summarize_condition_c()'s category_table loop
by_cat_lang = defaultdict(list)
for r in injected_rows:
    by_cat_lang[(r["metadata_invariant_category"], r["metadata_language_pair"])].append(r)

category_table = []
for cat in CATEGORY_MAP_INJECTED_TO_CHECKER:
    for pair in LANG_PAIRS:
        rows = by_cat_lang.get((cat, pair), [])
        attempted = [r for r in rows if r.get("metadata_status") not in ("hard_failed", "exception", "budget_exceeded_skip") and r.get("metadata_fixed") is not None]
        fixed = [r["metadata_fixed"] for r in attempted]
        regressed = [r["metadata_true_regression"] for r in attempted]
        cert = [r["metadata_c_certificate_achieved"] for r in attempted]
        entry = {
            "category": cat, "language_pair": pair, "n_rows": len(rows), "n_attempted": len(attempted),
            "fix_rate": round(sum(fixed) / len(fixed), 4) if fixed else None,
            "true_regression_rate": round(sum(regressed) / len(regressed), 4) if regressed else None,
            "certificate_rate": round(sum(cert) / len(cert), 4) if cert else None,
        }
        if cat == "negation_polarity_flip":
            n_localized = sum(1 for r in rows if r.get("metadata_negation_localized_pre_repair"))
            entry["negation_pre_repair_localization_rate"] = round(n_localized / len(rows), 4) if rows else None
        category_table.append(entry)

attempted_all = [r for r in injected_rows if r.get("metadata_status") not in ("hard_failed", "exception", "budget_exceeded_skip") and r.get("metadata_fixed") is not None]
fixed_all = [r["metadata_fixed"] for r in attempted_all]
regressed_all = [r["metadata_true_regression"] for r in attempted_all]
pooled_injected = {
    "n_rows": len(injected_rows), "n_attempted": len(attempted_all),
    "fix_rate": round(sum(fixed_all) / len(fixed_all), 4) if fixed_all else None,
    "true_regression_rate": round(sum(regressed_all) / len(regressed_all), 4) if regressed_all else None,
}

for entry in category_table:
    print(f"{entry['language_pair']:>10} | {entry['category']:<26} n={entry['n_rows']:<3} "
          f"fix_rate={entry['fix_rate']} true_regression_rate={entry['true_regression_rate']} "
          f"certificate_rate={entry['certificate_rate']}")
print()
print(f"Pooled over this {len(injected_rows)}-row injected subset: {pooled_injected}")


## Results

Certificate-check reproducibility, this subset's per-category fix/regression rates against the full 560-row run's pooled numbers (from `run_metadata["metrics_full_run"]`), and the comparison table against baselines B/D/C1/C2.

In [ ]:
# --- summary table -----------------------------------------------------------
full_run_incl = run_metadata["metrics_full_run"]["injected_pooled_incl_negation"]
full_run_excl = run_metadata["metrics_full_run"]["injected_pooled_excl_negation"]
comparison = run_metadata["comparison_table_B_D_C1_C2_vs_C"]["conditions"]

print("=" * 78)
print("CERTIFICATE-CHECK REPRODUCIBILITY (this notebook's checker.detect_all() vs the logged verdict)")
print("=" * 78)
print(f"  {n_cert_match}/{n_rows_checked} rows match exactly\n")

print("=" * 78)
print(f"THIS DEMO SUBSET ({len(injected_rows)} injected rows) vs FULL RUN (240 injected rows)")
print("=" * 78)
print(f"  {'metric':<28}{'this subset':>14}{'full run (incl. negation)':>28}{'full run (excl. negation)':>28}")
print(f"  {'fix_rate':<28}{pooled_injected['fix_rate']!s:>14}{full_run_incl['fix_rate']!s:>28}{full_run_excl['fix_rate']!s:>28}")
print(f"  {'true_regression_rate':<28}{pooled_injected['true_regression_rate']!s:>14}{full_run_incl['true_regression_rate']!s:>28}{full_run_excl['true_regression_rate']!s:>28}")

print()
print("=" * 78)
print("COMPARISON vs BASELINES (full 560-row runs, method_out.json comparison_table_B_D_C1_C2_vs_C)")
print("=" * 78)
for cond in ["B", "D", "C1", "C2", "C"]:
    row = comparison.get(cond)
    if not row:
        continue
    fix = row.get("fix_rate_pooled_excl_negation", row.get("fix_rate_pooled"))
    reg = row.get("true_regression_rate_pooled_excl_negation", row.get("true_regression_rate_pooled"))
    print(f"  {cond:<4} fix_rate={fix!s:<10} true_regression_rate={reg!s:<10}")

# --- plot ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cats = sorted(CATEGORY_MAP_INJECTED_TO_CHECKER.keys())
cat_fix = []
for cat in cats:
    rows_cat = [e for e in category_table if e["category"] == cat and e["fix_rate"] is not None]
    vals = [e["fix_rate"] for e in rows_cat]
    cat_fix.append(sum(vals) / len(vals) if vals else 0.0)

axes[0].bar(range(len(cats)), cat_fix, color="#4C72B0")
axes[0].set_xticks(range(len(cats)))
axes[0].set_xticklabels([c.replace("_", "\n") for c in cats], fontsize=8)
axes[0].set_ylabel("fix_rate (this demo subset)")
axes[0].set_ylim(0, 1)
axes[0].set_title("Fix rate by corruption category")

conds = [c for c in ["B", "D", "C1", "C2", "C"] if c in comparison]
fix_vals = [comparison[c].get("fix_rate_pooled_excl_negation", comparison[c].get("fix_rate_pooled")) for c in conds]
reg_vals = [comparison[c].get("true_regression_rate_pooled_excl_negation", comparison[c].get("true_regression_rate_pooled")) for c in conds]
x = range(len(conds))
width = 0.35
axes[1].bar([i - width / 2 for i in x], fix_vals, width, label="fix_rate", color="#4C72B0")
axes[1].bar([i + width / 2 for i in x], reg_vals, width, label="true_regression_rate", color="#C44E52")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(conds)
axes[1].set_ylim(0, 1)
axes[1].set_title("Condition C vs baselines (full 560-row runs)")
axes[1].legend()

plt.tight_layout()
plt.show()
